# RetainIQ — Phase 1.1: Dataset Overview

## Objective

Establish a clear understanding of the raw Telco customer dataset before applying any
cleaning or transformation.

This notebook answers four foundational questions:

1. How large is the dataset and what does one row represent?
2. What fields are available and how are they typed?
3. Can `Customer ID` act as the customer-level business key?
4. What is the high-level structure of the analytical population?


## 1. Environment Setup and Data Ingestion



In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Project convention:
# RetainIQ/
# ├── data/
# │   └── raw/
# │       └── telco.csv
# └── phase_01_data_audit/
#     └── notebooks/
#
# From this notebook, the raw dataset is two levels up from the phase folder.
DATA_PATH = Path(r"C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\telco.csv")

# Fallback for running the notebook alongside the uploaded file during development.
if not DATA_PATH.exists():
    DATA_PATH = Path("telco.csv")

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

Loaded: 7,043 rows × 50 columns


## 2. Dataset Size and Resource Footprint

Dataset dimensions provide the first boundary of the analytical problem. 

In [2]:
dataset_profile = pd.Series({
    "Rows": len(df),
    "Columns": df.shape[1],
    "Memory Usage (MB)": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "Exact Duplicate Rows": int(df.duplicated().sum())
})

dataset_profile

Rows                   7,043.00
Columns                   50.00
Memory Usage (MB)         11.92
Exact Duplicate Rows       0.00
dtype: float64

### Interpretation

The source file contains **7,043 customer records across 50 fields**.


## 3. Initial Record Inspection

A small sample is reviewed to understand the shape of a customer record before individual
fields are analyzed.

In [3]:
df.head(10)

,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,City,Zip Code,Latitude,Longitude,Population,Quarter,Referred a Friend,Number of Referrals,Tenure in Months,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,Internet Service,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,Los Angeles,90022,34.02,-118.16,68701,Q3,No,0,1,NaN,No,0.00,No,Yes,DSL,8,No,No,Yes,No,No,Yes,No,No,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,Los Angeles,90063,34.04,-118.19,55668,Q3,Yes,1,8,Offer E,Yes,48.85,Yes,Yes,Fiber Optic,17,No,Yes,No,No,No,No,No,Yes,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,"1,024.10",3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,Los Angeles,90065,34.11,-118.23,47534,Q3,No,0,18,Offer D,Yes,11.33,Yes,Yes,Fiber Optic,52,No,No,No,No,Yes,Yes,Yes,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,"1,752.55",45.61,0,203.94,"1,910.88",2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,Inglewood,90303,33.94,-118.33,27778,Q3,Yes,1,25,Offer C,Yes,19.76,No,Yes,Fiber Optic,12,No,Yes,Yes,No,Yes,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,98.50,"2,514.50",13.43,0,494.00,"2,995.07",2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,Whittier,90602,33.97,-118.02,26265,Q3,Yes,1,37,Offer C,Yes,6.33,Yes,Yes,Fiber Optic,14,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal,76.50,"2,868.15",0.00,0,234.21,"3,102.36",2,Churned,Yes,67,2793,Price,Extra data charges
5,4412-YLTKF,Female,72,No,Yes,No,Yes,1,United States,California,Pico Rivera,90660,33.99,-118.09,63288,Q3,No,0,27,Offer C,Yes,3.33,Yes,Yes,Fiber Optic,18,No,No,Yes,No,No,No,No,No,Month-to-Month,Yes,Bank Withdrawal,78.05,"2,135.50",0.00,10,89.91,"2,235.41",1,Churned,Yes,95,4638,Competitor,Competitor had better devices
6,0390-DCFDQ,Female,76,No,Yes,Yes,Yes,2,United States,California,Los Alamitos,90720,33.79,-118.07,21343,Q3,Yes,1,1,Offer E,Yes,15.28,No,Yes,Fiber Optic,30,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Mailed Check,70.45,70.45,0.00,0,15.28,85.73,2,Churned,Yes,76,3964,Other,Don't know
7,3445-HXXGF,Male,66,No,Yes,Yes,No,0,United States,California,Sierra Madre,91024,34.17,-118.06,10558,Q3,Yes,6,58,Offer B,No,0.00,No,Yes,DSL,24,No,Yes,Yes,No,No,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,45.30,"2,651.20",40.95,0,0.00,"2,610.25",1,Churned,Yes,91,5444,Dissatisfaction,Service dissatisfaction
8,2656-FMOKZ,Female,70,No,Yes,No,Yes,2,United States,California,Pasadena,91106,34.14,-118.13,23742,Q3,No,0,15,Offer D,Yes,44.07,Yes,Yes,Fiber Optic,19,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Mailed Check,74.45,"1,145.70",0.00,0,661.05,"1,806.75",2,Churned,Yes,91,5717,Dissatisfaction,Limited range of services
9,2070-FNEXE,Female,77,No,Yes,No,Yes,2,United States,California,Pasadena,91107,34.16,-118.09,32369,Q3,No,0,7,Offer E,Yes,26.95,No,Yes,Fiber Optic,18,Yes,No,No,No,No,No,No,No,Month-to-Month,No,Bank Withdrawal,76.45,503.60,11.05,0,188.65,681.20,2,Churned,Yes,81,4419,Price,Lack of affordable download/upload speed


## 4. Full Schema Inventory


In [4]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "null_pct": (df.isna().mean() * 100).round(2).values,
    "unique_values": df.nunique(dropna=True).values
})

schema

,column,dtype,non_null,null_count,null_pct,unique_values
0,Customer ID,object,7043,0,0.00,7043
1,Gender,object,7043,0,0.00,2
2,Age,int64,7043,0,0.00,62
3,Under 30,object,7043,0,0.00,2
4,Senior Citizen,object,7043,0,0.00,2
5,Married,object,7043,0,0.00,2
6,Dependents,object,7043,0,0.00,2
7,Number of Dependents,int64,7043,0,0.00,10
8,Country,object,7043,0,0.00,1
9,State,object,7043,0,0.00,1


## 5. Candidate Business Key and Analytical Grain

The intended grain is **one row per customer**.

`Customer ID` is therefore tested as the candidate business key. A valid customer-level key
should be non-null and unique across the dataset.

In [5]:
key_audit = pd.Series({
    "Row Count": len(df),
    "Unique Customer IDs": df["Customer ID"].nunique(dropna=True),
    "Missing Customer IDs": int(df["Customer ID"].isna().sum()),
    "Duplicate Customer IDs": int(df["Customer ID"].duplicated().sum()),
    "Customer ID Is Unique": bool(df["Customer ID"].is_unique)
})

key_audit

Row Count                 7043
Unique Customer IDs       7043
Missing Customer IDs         0
Duplicate Customer IDs       0
Customer ID Is Unique     True
dtype: object

In [6]:
assert df["Customer ID"].notna().all(), "Customer ID contains null values."
assert df["Customer ID"].is_unique, "Customer ID is not unique."

print("PASS — Customer ID is non-null and unique.")
print("Confirmed analytical grain: one row per customer.")

PASS — Customer ID is non-null and unique.
Confirmed analytical grain: one row per customer.


### Interpretation

`Customer ID` is a suitable customer-level key. There are **7,043 unique IDs across 7,043
records**, so no additional deduplication logic is required to establish the source grain.

## 6. Numeric vs. Categorical Field Inventory

This split provides an early view of how the data will later support statistical analysis,
segmentation, SQL dimensions, and machine-learning features.

In [7]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()

inventory = pd.DataFrame({
    "Field Type": ["Numeric", "Categorical / Text"],
    "Column Count": [len(numeric_cols), len(categorical_cols)]
})

inventory

,Field Type,Column Count
0,Numeric,19
1,Categorical / Text,31


## 7. Notebook 1 Conclusion

The source dataset has a well-defined customer-level grain, a unique `Customer ID`, and a
mixture of numeric and categorical attributes suitable for downstream analytics.

The next question is not *what can we calculate?* but **whether the raw fields are technically
reliable enough to support those calculations**.

**Next notebook:** `02_data_quality_audit.ipynb`